In [ ]:
# ROVER-style voting over four ASR systems (Faroese) with explicit ranking
# ------------------------------------------------------------------------------------
# This version runs locally and includes an optional meeting ID cap.
#
# To run, ensure your directory structure is:
# .
# ├── this_script.py
# ├── output/
# │   └── M1_modelA.jsonl
# │   └── ...
# ├── output2/
# │   └── M1_modelB.jsonl
# │   └── ...
# ├── output3/
# └── output4/
# ------------------------------------------------------------------------------------
!pip -q install rapidfuzz==3.9.7 jiwer==3.0.4 ujson==5.10.0

import os, re, glob, unicodedata, time
import ujson as ujson
from collections import defaultdict, Counter
from typing import Dict, List, Tuple, Any
from rapidfuzz.distance import Levenshtein
from jiwer import wer

# === 1) Environment Setup ===
# This script is designed to run in a local environment where the input directories
# are located relative to the script's execution path.

# === 2) CONFIG ===
BASE = "/work/FPSC/"  # Base path is the current directory.
INPUT_DIRS = ["output1", "output2", "output3", "output4"]  # Sub-folders under BASE
OUTPUT_FILE = os.path.join(BASE, "ROVER-voting/ROVER_voting_output_ALL-SYSTEMS.jsonl")

# Optional: turn on model weighting (if True, apply WEIGHTS below)
USE_MODEL_WEIGHTS = True

# Optional: Limit processing to meetings up to a certain ID.
# If True, the script will only process files from meetings where the
# numeric ID is less than or equal to LIMIT_MEETING_ID.
USE_MEETING_ID_LIMIT = True
LIMIT_MEETING_ID = 6690  # Process meetings with ID less than or equal <= this value (e.g., M6606).

# Map model-id -> short name
MODEL_SHORT = {
    "davidilag/wav2vec2-xls-r-300m-cpt-1000h_faroese-cp_best-faroese-100h-60-epochs_run8_2025-09-24": "Wav2Vec2-FO-CPT",
    "davidilag/wav2vec2-xls-r-300m-faroese-100h-60-epochs_20250122": "Wav2Vec2-FO",
    "carlosdanielhernandezmena/whisper-large-faroese-8k-steps-100h": "Whisper-FO",
    "davidilag/whisper-large-no-is-fo-100h-30k-steps": "Whisper-NO/IS/FO",
}

# Higher weight = we trust this model more in close calls
WEIGHTS = {
    "Wav2Vec2-CPT-FO": 1.04,        # strongest (continual pretraining)
    "Whisper-NO/IS/FO": 1.08,       # multilingual whisper
    "Whisper-FO": 0.93,             # Faroese-only whisper
    "Wav2Vec2-FO": 0.95             # Faroese-only 
}

# === 3) Helpers ===

_ws = re.compile(r"\s+")

def normalize_text(s: str) -> str:
    """
    Keep Faroese diacritics, normalize width/case/spacing.
    """
    if s is None:
        return ""
    s = unicodedata.normalize("NFKC", s)
    s = s.strip().lower()
    s = _ws.sub(" ", s)
    return s

def tokenize(s: str) -> List[str]:
    # Simple whitespace tokenization after normalization
    return normalize_text(s).split()

def norm_token_levenshtein(a_tokens: List[str], b_tokens: List[str]) -> float:
    """
    Normalized Levenshtein over tokens (0.0 perfect match, 1.0 maximally different).
    """
    return Levenshtein.normalized_distance(a_tokens, b_tokens)

def medoid_by_weighted_distance(candidates: List[Tuple[str, str, List[str], float]]) -> Tuple[int, float, List[float]]:
    """
    Pick index of candidate minimizing weighted sum of distances to others.
    candidates: list of (model_id, short_name, tokens, weight)
    Returns: (best_index, min_cost, costs_per_candidate)
    """
    n = len(candidates)
    costs = [0.0] * n
    for i in range(n):
        _, _, ti, wi = candidates[i]
        total = 0.0
        for j in range(n):
            if i == j:
                continue
            _, _, tj, wj = candidates[j]
            d = norm_token_levenshtein(ti, tj)
            total += d * wj
        costs[i] = total
    k = min(range(n), key=lambda idx: costs[idx])
    return k, costs[k], costs

def agreement_confidence(all_tokens: List[List[str]], winner_idx: int) -> float:
    """
    Turn relative agreement into a [0,1] confidence.
    1.0 if at least two identical candidates (handled earlier),
    else 1 - (avg normalized distance to others), bounded to [0.5, 0.95].
    """
    n = len(all_tokens)
    if n <= 1:
        return 1.0
    tw = all_tokens[winner_idx]
    dists = []
    for j, t in enumerate(all_tokens):
        if j == winner_idx:
            continue
        dists.append(norm_token_levenshtein(tw, t))
    avg_d = sum(dists) / len(dists) if dists else 0.0
    raw = 1.0 - avg_d
    return max(0.5, min(0.95, raw))

def read_jsonl(path: str):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                yield ujson.loads(line)
            except Exception:
                continue  # skip malformed line

def short_model_name(model_id: str) -> str:
    return MODEL_SHORT.get(model_id, model_id.split("/")[-1])

def get_meeting_id_from_filename(fn: str) -> str:
    # e.g., "M12_wav2vec2-cpt-fo.jsonl" -> "M12"
    base = os.path.basename(fn)
    return base.split("_")[0]

def _build_ranking(entries, majority_norm_text=None):
    """
    entries: [(model_id, short_name, tokens, weight), ...]
    majority_norm_text: if provided, marks models whose normalized text == majority as winners (identical_plurality case)
    returns: (ranking_list, costs_list)
    """
    n = len(entries)
    costs = [0.0] * n
    for i in range(n):
        _, _, ti, wi = entries[i]
        total = 0.0
        for j in range(n):
            if i == j:
                continue
            _, _, tj, wj = entries[j]
            d = norm_token_levenshtein(ti, tj)
            total += d * wj
        costs[i] = total

    # ranks: 1 = best (allow ties)
    sorted_costs = sorted(set(costs))
    rank_map = {c: (idx + 1) for idx, c in enumerate(sorted_costs)}

    ranking = []
    for i, (m, s, t, w) in enumerate(entries):
        norm_txt = " ".join(t)
        is_winner = False
        if majority_norm_text is not None:
            is_winner = (norm_txt == majority_norm_text)
        ranking.append({
            "model_id": m,
            "model_short": s,
            "text": norm_txt,
            "weight": w,
            "cost": costs[i],
            "rank": rank_map[costs[i]],
            "is_winner": is_winner,
        })
    ranking.sort(key=lambda x: (x["rank"], x["cost"], x["model_short"]))
    return ranking, costs

# === 4) Ingest: build {audio_id: {short_model_name: transcript_text, ...}} ===

def collect_candidates() -> Dict[str, Dict[str, Dict[str, Any]]]:
    """
    Return:
      audio_to_cands[audio_id][short_name] = {
        'text': full_text, 'num_words': int, 'model_id': model_id, 'meeting_id': 'MXXXX',
        'file': file_path
      }
    """
    audio_to_cands: Dict[str, Dict[str, Dict[str, Any]]] = defaultdict(dict)

    for d in INPUT_DIRS:
        dpath = os.path.join(BASE, d)
        files = sorted(glob.glob(os.path.join(dpath, "M*_*.jsonl")))
        if not files:
            print(f"⚠️ No files found in {dpath}")
        for fp in files:
            meeting_id = get_meeting_id_from_filename(fp)

            # --- NEW: Optional meeting ID cap ---
            if USE_MEETING_ID_LIMIT:
                try:
                    # Extract numeric part of the meeting ID, e.g., 'M6606' -> 6606
                    numeric_match = re.search(r'\d+', meeting_id)
                    if numeric_match:
                        numeric_meeting_id = int(numeric_match.group())
                        if numeric_meeting_id > LIMIT_MEETING_ID:
                            continue # Skip this file
                    else:
                        print(f"⚠️ Could not parse numeric ID from meeting '{meeting_id}'. Skipping file {fp}.")
                        continue
                except (AttributeError, ValueError):
                    print(f"⚠️ Error parsing meeting ID for file {fp}. Skipping.")
                    continue
            # --- End of new code ---

            for obj in read_jsonl(fp):
                model_id = obj.get("model") or obj.get("model_id") or ""
                sname = short_model_name(model_id)
                audio_id = obj.get("audio_id")
                tinfo = obj.get("transcript", {})
                text = tinfo.get("full_text", "")
                n_words = tinfo.get("num_words", None)
                if isinstance(n_words, (int, float)):
                    n_words = int(n_words)
                else:
                    n_words = len(text.split()) if text else 0

                if not audio_id or not text:
                    continue
                prev = audio_to_cands[audio_id].get(sname)
                if (prev is None) or ((prev.get("num_words") or 0) < n_words):
                    audio_to_cands[audio_id][sname] = {
                        "text": text,
                        "num_words": n_words,
                        "model_id": model_id,
                        "meeting_id": meeting_id,
                        "file": fp,
                    }
    return audio_to_cands

# === 5) Voting loop (stream to JSONL) ===

def run_voting(audio_to_cands: Dict[str, Dict[str, Dict[str, Any]]],
               out_path: str,
               use_weights: bool = True):
    if os.path.exists(out_path):
        print(f"⚠️ Output exists, will OVERWRITE: {out_path}")
        os.remove(out_path)

    stats_wins = Counter()
    stats_identical = 0
    total = 0

    start = time.time()
    with open(out_path, "a", encoding="utf-8") as fout:
        # Sort by audio_id for deterministic output
        sorted_audio_ids = sorted(audio_to_cands.keys())

        for audio_id in sorted_audio_ids:
            cand_map = audio_to_cands[audio_id]
            # We only vote when we have at least 2 candidates; ideally 4
            if len(cand_map) < 2:
                continue

            total += 1
            entries = []
            for sname, meta in cand_map.items():
                tokens = tokenize(meta["text"])
                w = WEIGHTS.get(sname, 1.0) if use_weights else 1.0
                entries.append((meta["model_id"], sname, tokens, w))

            # Exact/plurality identical check by normalized text
            norm_texts = [" ".join(t) for (_, _, t, _) in entries]
            counts = Counter(norm_texts)
            top_text, top_cnt = counts.most_common(1)[0]

            if top_cnt >= 2:
                # representative winner (first which matches top_text)
                winner_idx = norm_texts.index(top_text)
                win_model_id, win_short, win_tokens, _ = entries[winner_idx]

                # Build ranking (costs + ranks) and mark identical winners
                ranking, costs_list = _build_ranking(entries, majority_norm_text=top_text)

                stats_identical += 1
                stats_wins[win_short] += 1
                out = {
                    "audio_id": audio_id,
                    "meeting_id": cand_map[win_short]["meeting_id"],
                    "vote_type": "identical_plurality",
                    "confidence": 1.0,
                    "winner_model_id": win_model_id,
                    "winner_model_short": win_short,
                    "winner_text": " ".join(win_tokens),
                    "costs": { r["model_short"]: r["cost"] for r in ranking },
                    "ranking": ranking,  # includes cost, rank, is_winner
                    "candidates": [
                        {"model_id": m, "model_short": s, "text": " ".join(t), "weight": w}
                        for (m, s, t, w) in entries
                    ],
                }
                fout.write(ujson.dumps(out, ensure_ascii=False) + "\n")
                continue

            # Otherwise: weighted-medoid by pairwise token Levenshtein
            k, min_cost, all_costs = medoid_by_weighted_distance(entries)
            win_model_id, win_short, win_tokens, _ = entries[k]
            stats_wins[win_short] += 1

            conf = agreement_confidence([t for (_,_,t,_) in entries], k)

            # Build ranking; set is_winner only for the medoid in this branch
            ranking, _ = _build_ranking(entries, majority_norm_text=None)
            for r in ranking:
                r["is_winner"] = (r["model_short"] == win_short)

            out = {
                "audio_id": audio_id,
                "meeting_id": cand_map[win_short]["meeting_id"],
                "vote_type": "weighted_medoid",
                "confidence": conf,
                "winner_model_id": win_model_id,
                "winner_model_short": win_short,
                "winner_text": " ".join(win_tokens),
                "costs": { r["model_short"]: r["cost"] for r in ranking },
                "ranking": ranking,  # includes cost, rank, is_winner
                "candidates": [
                    {"model_id": m, "model_short": s, "text": " ".join(t), "weight": w}
                    for (m, s, t, w) in entries
                ],
            }
            fout.write(ujson.dumps(out, ensure_ascii=False) + "\n")

    secs = time.time() - start
    print("\n========== ROVER VOTING COMPLETE ==========")
    print(f"Decisions written: {total:,}")
    print(f"Identical pluralities: {stats_identical:,} ({(stats_identical/max(total,1))*100:.1f}%)")
    print("\nWins by model:")
    for k, v in stats_wins.most_common():
        print(f"  {k:20s} : {v:,}  ({(v/max(total,1))*100:.1f}%)")
    print(f"\n✅ Output -> {out_path}")
    print(f"⏱️ Wall time: {secs:.1f}s")

# === 6) Run ===
print("🔍 Collecting candidates from input files...")
audio_to_cands = collect_candidates()
print(f"Found audio_ids with >=1 candidate: {len(audio_to_cands):,}")
filtered = {aid: c for aid, c in audio_to_cands.items() if len(c) >= 2}
print(f"Audio_ids with >=2 systems: {len(filtered):,}")

if not filtered:
    print("\n⚠️ No audio segments had at least two candidates. Nothing to vote on. Exiting.")
else:
    run_voting(filtered, OUTPUT_FILE, use_weights=USE_MODEL_WEIGHTS)

🔍 Collecting candidates from input files...
Found audio_ids with >=1 candidate: 17,529
Audio_ids with >=2 systems: 17,396

========== ROVER VOTING COMPLETE ==========
Decisions written: 17,396
Identical pluralities: 1,167 (6.7%)

Wins by model:
  Wav2Vec2-FO-CPT      : 10,948  (62.9%)
  Whisper-NO/IS/FO     : 3,119  (17.9%)
  Wav2Vec2-FO          : 1,797  (10.3%)
  Whisper-FO           : 1,532  (8.8%)

✅ Output -> /work/FPSC/ROVER-voting/ROVER_voting_output_ALL-SYSTEMS_Cap-M6690.jsonl
⏱️ Wall time: 19.5s
